In [5]:
#Import the libraries and read files 

import warnings; warnings.filterwarnings("ignore")
import os
import numpy as np, pandas as pd
 
#Read the file
 
#Dataset 1:
Brics_merged=pd.read_csv("BRICS_merged.csv")
 
#Dataset 2:
Development_merged=pd.read_csv("development_merged_cleaned.csv")
 
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression, chi2, RFE
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import r2_score, mean_absolute_error, f1_score, accuracy_score, classification_report
 
os.makedirs("out", exist_ok=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
pd.options.display.float_format = "{:,.2f}".format
 
 
def heat(corr, title, path):
    fig, ax = plt.subplots(figsize=(9, 7.5))
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=60, ha="right")
    ax.set_yticks(range(len(corr.index))); ax.set_yticklabels(corr.index)
    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            ax.text(j, i, f"{corr.iat[i, j]:.2f}", ha="center", va="center", fontsize=7)
    fig.colorbar(im); ax.set_title(title); fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)

In [6]:
#DATASET 2: DEVELOPMENT 
print("=" * 30, "DEVELOPMENT DATASET", "=" * 30)
dev = Development_merged.copy()
short = {
    "Access to electricity (% of population)": "electricity",
    "Renewable energy consumption (% of total final energy consumption)": "renewable_energy",
    "Individuals using the Internet (% of population)": "internet_users",
    "Agriculture, forestry, and fishing, value added (% of GDP)": "agriculture_gdp",
    "GDP growth (annual %)": "gdp_growth",
    "GDP per capita (constant 2015 US$)": "gdp_per_capita",
    "Current health expenditure (% of GDP)": "health_spend",
    "Mortality rate, infant (per 1,000 live births)": "infant_mortality",
    "Life expectancy at birth, total (years)": "life_expectancy",
    "Fertility rate, total (births per woman)": "fertility",
    "Population, total": "population",
    "Urban population (% of total population)": "urban_pop",
}
# long -> wide: one row per country-year, one column per indicator
wide = dev.pivot(index=["country_name", "year"], columns="indicator_name", values="value").rename(columns=short).reset_index()
num = wide.drop(columns=["country_name", "year"])
print("countries:", sorted(wide.country_name.unique()))
print("wide shape:", wide.shape, "\nmissing values per indicator:\n", num.isna().sum().to_string())
 
stats = pd.DataFrame({"mean": num.mean(), "median": num.median(), "mode": num.mode().iloc[0],
                      "std": num.std(), "skew": num.skew()})
print("\nMean / median / mode:\n", stats)
print("\nCountry averages:\n", wide.groupby("country_name")[list(num.columns)].mean().T)
 
# plots
fig, axs = plt.subplots(3, 4, figsize=(16, 9))
for ax, c in zip(axs.ravel(), num.columns):
    ax.hist(num[c].dropna(), bins=25, color="#4C78A8"); ax.set_title(c, fontsize=10)
fig.tight_layout(); fig.savefig("out/dev_histograms.png", dpi=130); plt.close(fig)
 
fig, ax = plt.subplots(figsize=(9, 5))
for c, g in wide.groupby("country_name"):
    ax.plot(g.year, g.life_expectancy, label=c)
ax.set_title("Life expectancy over time"); ax.set_ylabel("years"); ax.legend(fontsize=8, ncol=2)
fig.tight_layout(); fig.savefig("out/dev_life_expectancy_trend.png", dpi=130); plt.close(fig)
 
# Spearman: indicators are skewed (population, GDP per capita), so rank correlation is safer than Pearson
corr = num.corr(method="spearman")
heat(corr, "Spearman correlation - development indicators", "out/dev_correlation.png")
print("\nCorrelation with life_expectancy:\n", corr["life_expectancy"].drop("life_expectancy").sort_values().to_string())
 
# ---- ML: predict life expectancy; train on 2000-2018, test on 2019-2025 (no future leakage) ----
target = "life_expectancy"
feats = [c for c in num.columns if c != target]
d = wide.dropna(subset=[target]).copy()
d["population"] = np.log10(d["population"])
tr, te = d[d.year <= 2018], d[d.year > 2018]
print(f"\ntrain rows {len(tr)} (<=2018), test rows {len(te)} (>2018)")
 
 
def build(model, sel, k):
    steps = [("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]  # fit on train only
    if sel == "SelectKBest": steps.append(("sel", SelectKBest(f_regression, k=k)))
    if sel == "RFE": steps.append(("sel", RFE(LinearRegression(), n_features_to_select=k)))
    return Pipeline(steps + [("m", model)])
 
 
models = {"Linear": LinearRegression, "RandomForest": lambda: RandomForestRegressor(300, random_state=0, n_jobs=-1)}
 
 
def compare(fs, label):
    rows = []
    for mn, mk in models.items():
        for sel, k in [("none (all)", len(fs))] + [(s, k) for s in ("SelectKBest", "RFE") for k in (3, 5, 8) if k < len(fs)]:
            p = build(mk(), "none" if sel.startswith("none") else sel, k).fit(tr[fs], tr[target])
            pr = p.predict(te[fs])
            rows.append([label, mn, sel, k, r2_score(te[target], pr), mean_absolute_error(te[target], pr)])
    return pd.DataFrame(rows, columns=["features", "model", "selection", "k", "R2", "MAE_years"])
 
 
res = pd.concat([compare(feats, "all 11"), compare([f for f in feats if f != "infant_mortality"], "no infant_mortality")])
print("\nModel comparison on held-out years:\n", res.to_string(index=False))
 
# which features each selector keeps (k=5)
imp = SimpleImputer(strategy="median").fit(tr[feats]); Xtr = pd.DataFrame(imp.transform(tr[feats]), columns=feats)
Xs = StandardScaler().fit_transform(Xtr)
print("\nSelectKBest(f_regression, k=5):", [f for f, m in zip(feats, SelectKBest(f_regression, k=5).fit(Xs, tr[target]).get_support()) if m])
print("RFE(LinearRegression, k=5):   ", [f for f, m in zip(feats, RFE(LinearRegression(), n_features_to_select=5).fit(Xs, tr[target]).get_support()) if m])
rf = RandomForestRegressor(300, random_state=0, n_jobs=-1).fit(Xtr, tr[target])
fi = pd.Series(rf.feature_importances_, index=feats).sort_values()
print("\nRandomForest importances:\n", fi.sort_values(ascending=False).to_string())
fig, ax = plt.subplots(figsize=(7, 5)); fi.plot.barh(ax=ax, color="#4C78A8"); ax.set_title("RF feature importance (life expectancy)")
fig.tight_layout(); fig.savefig("out/dev_feature_importance.png", dpi=130); plt.close(fig)
 
pr = rf.predict(pd.DataFrame(imp.transform(te[feats]), columns=feats))
fig, ax = plt.subplots(figsize=(5.5, 5.5)); ax.scatter(te[target], pr, s=14)
lo, hi = te[target].min(), te[target].max(); ax.plot([lo, hi], [lo, hi], "r--")
ax.set_xlabel("actual"); ax.set_ylabel("predicted"); ax.set_title("RF: held-out years")
fig.tight_layout(); fig.savefig("out/dev_actual_vs_pred.png", dpi=130); plt.close(fig)

============================== DEVELOPMENT DATASET ==============================
countries: ['Afghanistan', 'Bangladesh', 'Bhutan', 'India', 'Maldives', 'Nepal', 'Pakistan', 'Sri Lanka']
wide shape: (208, 14) 
missing values per indicator:
 indicator_name
electricity          8
agriculture_gdp      6
health_spend        18
fertility            8
gdp_growth           2
gdp_per_capita       1
internet_users      12
life_expectancy      8
infant_mortality     8
population           0
renewable_energy    29
urban_pop            0

Mean / median / mode:
                            mean        median       mode            std  skew
indicator_name                                                               
electricity               78.05         86.95     100.00          22.47 -1.12
agriculture_gdp           18.01         17.58       3.84           8.23  0.33
health_spend               5.03          3.82       1.77           3.43  2.20
fertility                  3.04          2.38       2

In [7]:
# ============================ DATASET 1: BRICS ============================
print("\n" + "=" * 30, "BRICS DATASET", "=" * 30)
b = Brics_merged.copy()
print("truncated / damaged paragraphs:", int(b.text_truncated.sum()), int(b.text_encoding_damaged.sum()))
print("encoding-damaged paragraphs by year:\n", b.groupby("year").text_encoding_damaged.sum().to_string())
b = b[~b.text_encoding_damaged.astype(bool)].copy()
print("paragraphs used:", len(b), "| summits:", b.year.nunique())
 
cols = ["word_count", "topic_count", "member_count", "partner_count", "participant_count"]
print("\nMean / median / mode:\n", pd.DataFrame({"mean": b[cols].mean(), "median": b[cols].median(), "mode": b[cols].mode().iloc[0]}))
print("\nPer summit year:\n", b.groupby("year").agg(paragraphs=("paragraph_id", "count"), mean_words=("word_count", "mean"),
                                                   members=("member_count", "first"), partners=("partner_count", "first")))
 
t = b.assign(topic=b.topics.fillna("").str.split(" | ", regex=False)).explode("topic")
t = t[t.topic != ""]
freq = t.topic.value_counts()
print("\nTopic frequency (paragraph mentions):\n", freq.to_string())
 
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
b.groupby("year").size().plot.bar(ax=axs[0], color="#4C78A8"); axs[0].set_title("Paragraphs per summit declaration")
b.groupby("year").word_count.mean().plot(ax=axs[1], marker="o"); axs[1].set_title("Mean paragraph length (words)")
freq.sort_values().plot.barh(ax=axs[2], color="#F58518"); axs[2].set_title("Topic frequency")
fig.tight_layout(); fig.savefig("out/brics_overview.png", dpi=130); plt.close(fig)
 
share = pd.crosstab(t.year.values, t.topic.values).div(b.groupby("year").size(), axis=0)
fig, ax = plt.subplots(figsize=(11, 6)); im = ax.imshow(share.T, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(share.index))); ax.set_xticklabels(share.index, rotation=60)
ax.set_yticks(range(len(share.columns))); ax.set_yticklabels(share.columns, fontsize=8)
fig.colorbar(im, label="share of paragraphs"); ax.set_title("Topic emphasis by summit year")
fig.tight_layout(); fig.savefig("out/brics_topic_by_year.png", dpi=130); plt.close(fig)
 
bc = b[["year"] + cols].corr()
heat(bc, "BRICS numeric correlation", "out/brics_correlation.png")
print("\nCorrelation:\n", bc)
 
# ---- ML: does the paragraph text identify its most common topic? TF-IDF -> chi2 selection -> logistic regression ----
top = freq.index[0]
b["y"] = b.topics.fillna("").str.contains(top, regex=False).astype(int)
btr, bte = b[b.year <= 2022], b[b.year > 2022]  # train on earlier summits, test on later ones
print(f"\nTarget: paragraph tagged '{top}' | train {len(btr)} (<=2022), test {len(bte)} | positive rate train {btr.y.mean():.2f}, test {bte.y.mean():.2f}")
print("Majority-class baseline accuracy:", round(max(bte.y.mean(), 1 - bte.y.mean()), 3))
for k in (50, 200, 1000, "all"):
    p = Pipeline([("tf", TfidfVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2))),
                  ("sel", SelectKBest(chi2, k=k)),
                  ("lr", LogisticRegression(class_weight="balanced", max_iter=1000))]).fit(btr.paragraph_text, btr.y)
    pr = p.predict(bte.paragraph_text)
    print(f"k={k:>5}: accuracy {accuracy_score(bte.y, pr):.3f}  F1 {f1_score(bte.y, pr):.3f}")
tf = TfidfVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2)).fit(btr.paragraph_text)
Xb = tf.transform(btr.paragraph_text); sc = pd.Series(chi2(Xb, btr.y)[0], index=tf.get_feature_names_out()).sort_values(ascending=False)
print("\nTop chi2 terms for the topic:\n", sc.head(15).to_string())
print("\n", classification_report(bte.y, pr))


============================== BRICS DATASET ==============================
truncated / damaged paragraphs: 1 305
encoding-damaged paragraphs by year:
 year
2009     0
2010     0
2011     0
2012     0
2013    12
2014    17
2015    16
2016    22
2017     0
2018    20
2019     0
2020    25
2021    28
2022    23
2023    16
2024    33
2025    49
2026    44
paragraphs used: 1118 | summits: 18

Mean / median / mode:
                    mean  median  mode
word_count        92.12   80.00 39.00
topic_count        2.29    2.00  1.00
member_count       6.09    5.00  5.00
partner_count      2.01    0.00  0.00
participant_count  6.18    5.00  5.00

Per summit year:
       paragraphs  mean_words  members  partners
year                                           
2009          16       56.94        4         0
2010          33       71.61        4         0
2011          32       51.38        5         0
2012          50       71.52        5         0
2013          35       80.80        5         0
2

In [8]:
#Life expectancy gain per countr 
g = wide.sort_values("year").groupby("country_name")
print((g.life_expectancy.last() - g.life_expectancy.first()).sort_values(ascending=False).rename("life_exp_gain_years"))

country_name
Bangladesh    12.88
Bhutan        11.42
Afghanistan   11.28
Maldives      10.39
India          9.49
Nepal          7.99
Sri Lanka      6.49
Pakistan       5.90
Name: life_exp_gain_years, dtype: float64


In [9]:
#Topic co-occurrence (which topics appear together) 
Xv = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(d[feats]), columns=feats)
vif = {c: 1 / (1 - LinearRegression().fit(Xv.drop(columns=c), Xv[c]).score(Xv.drop(columns=c), Xv[c])) for c in feats}
print(pd.Series(vif).sort_values(ascending=False))

fertility          13.80
infant_mortality   13.22
gdp_per_capita      6.65
agriculture_gdp     6.35
renewable_energy    4.53
population          4.27
health_spend        3.62
electricity         2.96
urban_pop           2.57
internet_users      2.14
gdp_growth          1.11
dtype: float64


In [12]:
#Topic co-occurrence
T = pd.DataFrame({k: b.topics.fillna("").str.contains(k, regex=False) for k in freq.index}).astype(int)
co = T.T @ T
print(co.loc[freq.index[:8], freq.index[:8]])

topic                                            Political and security affairs  Global governance and multilateral reform  Science, technology and innovation  \
topic                                                                                                                                                            
Political and security affairs                                              294                                        125                                  59   
Global governance and multilateral reform                                   125                                        289                                  49   
Science, technology and innovation                                           59                                         49                                 260   
Peace, conflict and international security                                  162                                         99                                  28   
Trade and investment        

In [13]:
#Before vs after expansion (2024 onward), with a chi-square test per topic
#Science and technology, digital/AI, climate, infrastructure, SDGs and trade all rise significantly after 2024. Health and culture do not change significantly.

from scipy.stats import chi2_contingency
new = b.year >= 2024
out = {k: chi2_contingency(pd.crosstab(new, T[k]))[1] for k in freq.index}
print(pd.DataFrame({"share_old": T[~new].mean(), "share_new": T[new].mean(), "p_value": pd.Series(out)}).sort_values("p_value").head(8))

                                                share_old  share_new  p_value
Science, technology and innovation                   0.19       0.36     0.00
Digital economy / ICT / AI                           0.05       0.16     0.00
Climate and environment                              0.11       0.22     0.00
Development and infrastructure                       0.06       0.14     0.00
Sustainable development / SDGs                       0.14       0.22     0.00
Trade and investment                                 0.14       0.22     0.00
Health                                               0.07       0.10     0.12
Culture / youth / people-to-people cooperation       0.08       0.11     0.16


In [14]:
#10. Predict paragraph length (regression)
#This only reached R² 0.13, so these numeric features barely explain paragraph length. It works as a “negative result” to report.

from sklearn.ensemble import RandomForestRegressor as RFR
Xn = b[["year", "topic_count", "member_count", "partner_count"]]
m = RFR(200, random_state=0).fit(Xn[b.year <= 2022], b.word_count[b.year <= 2022])
print("R2:", r2_score(b.word_count[b.year > 2022], m.predict(Xn[b.year > 2022])))

R2: 0.13241606521426008
